# 09 — Live Deployment
## Generate H2H Matchup Predictions for Upcoming Tournaments

**Prerequisites:**
- All validation gates passed (Notebook 08)
- Model calibrated on full available data
- DataGolf API key configured

**Workflow:**
1. Load all historical rounds (training data)
2. Define target tournament and get field
3. Compute EWMA skill estimates + course fit + recent form
4. Run MC simulation with `compute_h2h=True` (50K sims)
5. Load live matchup odds from sportsbook
6. Detect edges: model P(A>B) vs market P(A>B)
7. Size bets via fractional Kelly
8. Log and track bets

In [ ]:
# --- Setup ---
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from datetime import datetime

from config.settings import Settings
from data.loader import DataLoader
from features.time_weighting import TimeWeighter
from simulation.monte_carlo import MonteCarloSimulator, PlayerAbility
from simulation.tournament import TournamentConfig
from betting.kelly import kelly_fraction
from betting.bankroll import BankrollTracker

cfg = Settings()
# EWMA 25/120 and PROB_TEMPERATURE 0.35 are now in settings.py defaults

# ── YOUR SPORTSBOOK ACCOUNTS ──
# Only books you have accounts with. Set to None to use ALL books.
MY_BOOKS = ["draftkings", "pinnacle", "betmgm"]

In [ ]:
# --- Step 1: Load All Historical Rounds (training + holdout + 2025 + 2026) ---
# For live deployment, use ALL available data — no holdout restriction.
loader = DataLoader(cfg)
rounds_df = loader.load_rounds()

# Also load holdout + newer season rounds from subfolders and extra files
extra_files = [
    cfg.DATA_DIR / "holdout" / "sg_rounds_pga_2023_2024.csv",
    cfg.DATA_DIR / "sg_rounds_pga_2025.csv",
    cfg.DATA_DIR / "sg_rounds_pga_2026.csv",
]
for fpath in extra_files:
    if fpath.exists():
        extra = pd.read_csv(fpath, low_memory=False)
        rounds_df = pd.concat([rounds_df, extra], ignore_index=True)
        print(f"Added {len(extra):,} rounds from {fpath.name}")

# Normalize: rename dg_id → player_id (loader returns dg_id from CSV column names)
if "dg_id" in rounds_df.columns and "player_id" not in rounds_df.columns:
    rounds_df = rounds_df.rename(columns={"dg_id": "player_id"})
if "date" not in rounds_df.columns and "event_completed" in rounds_df.columns:
    rounds_df["date"] = pd.to_datetime(rounds_df["event_completed"], errors="coerce")

rounds_df["date"] = pd.to_datetime(rounds_df["date"])
rounds_df = rounds_df.sort_values("date").reset_index(drop=True)

print(f"\nTotal rounds: {len(rounds_df):,}")
print(f"Players: {rounds_df['player_id'].nunique():,}")
print(f"Date range: {rounds_df['date'].min().date()} → {rounds_df['date'].max().date()}")

In [ ]:
# --- Step 2: Compute Player Skill Estimates ---
weighter = TimeWeighter(cfg)
player_features = weighter.compute_weighted_sg(rounds_df, as_of_date=pd.Timestamp.now())

print(f"Skill estimates for {len(player_features)} players")
print(f"Top 10 by EWMA SG Total:")
top10 = player_features.nlargest(10, "ewma_sg_total")[["player_id", "ewma_sg_total"]]
print(top10.to_string(index=False))

In [ ]:
# --- Step 3: Define Target Tournament & Get Field ---
import json, urllib.request

# Get field from DataGolf pre-tournament predictions (current week)
url = (f"{cfg.DATAGOLF_BASE_URL}/preds/pre-tournament"
       f"?tour=pga&odds_format=decimal&file_format=json&key={cfg.DATAGOLF_API_KEY}")

req = urllib.request.Request(url, headers={"User-Agent": "golf-model/1.0"})
resp = urllib.request.urlopen(req, timeout=30)
field_data = json.loads(resp.read().decode("utf-8"))

api_event = field_data.get("event_name", "Unknown")
print(f"API is currently serving: {api_event}")

# Auto-detect event_id: try known IDs first, then schedule lookup
TARGET_EVENT_NAME = api_event
TARGET_EVENT_ID = None

KNOWN_IDS = {
    "players": 11, "arnold palmer": 9, "masters": 14, "us open": 26,
    "open championship": 27, "pga championship": 28, "memorial": 16,
    "genesis": 7, "waste management": 5, "sentry": 1, "farmers": 3,
    "at&t pebble": 4, "honda": 8, "valspar": 10, "texas open": 12,
    "wells fargo": 15, "byron nelson": 17, "charles schwab": 18,
    "rlx technology": 25, "travelers": 19, "rocket mortgage": 20,
    "john deere": 21, "barbasol": 22, "wyndham": 24,
    "fedex st. jude": 29, "bmw": 30, "tour championship": 31,
}
for key, eid in KNOWN_IDS.items():
    if key in api_event.lower():
        TARGET_EVENT_ID = eid
        break

# Fallback: schedule file lookup (use longest matching word, skip short words)
if TARGET_EVENT_ID is None:
    search_words = [w for w in api_event.split() if len(w) > 3]
    search_term = " ".join(search_words[:2]) if search_words else api_event
    for sched_file in ["schedule_2026.csv", "schedule_2025.csv"]:
        sched_path = cfg.DATA_DIR / sched_file
        if not sched_path.exists():
            continue
        sched = pd.read_csv(sched_path)
        name_col = [c for c in sched.columns if "event_name" in c.lower() or "name" in c.lower()]
        id_col = [c for c in sched.columns if "event_id" in c.lower() or c == "dg_id"]
        if name_col and id_col:
            match = sched[sched[name_col[0]].str.contains(search_term, case=False, na=False)]
            if len(match) > 0:
                TARGET_EVENT_ID = int(match.iloc[0][id_col[0]])
                break

if TARGET_EVENT_ID is None:
    TARGET_EVENT_ID = 0
    print(f"⚠ Could not determine event_id — set TARGET_EVENT_ID manually")

# Extract player IDs — use baseline_history_fit (includes course history adjustment)
dg_field = field_data.get("baseline_history_fit", field_data.get("baseline", []))

def _split_team_name(team_name):
    """Split a team_name like \"M. Fitzpatrick / A. Fitzpatrick\" into two names."""
    if not team_name:
        return "", ""
    parts = [s.strip() for s in team_name.split("/")]
    if len(parts) == 2:
        return parts[0], parts[1]
    return team_name, ""

def _extract_field(entries):
    """Handle both individual events (dg_id/player_name) and team events (p1_dg_id/p2_dg_id + team_name)."""
    ids, names = [], {}
    for p in entries:
        if "dg_id" in p:
            pid = int(p["dg_id"])
            ids.append(pid)
            names[pid] = p.get("player_name", "")
        elif "p1_dg_id" in p or "p2_dg_id" in p:
            n1, n2 = _split_team_name(p.get("team_name", ""))
            for prefix, nm in (("p1_", n1), ("p2_", n2)):
                id_key = f"{prefix}dg_id"
                if p.get(id_key) is not None:
                    pid = int(p[id_key])
                    ids.append(pid)
                    names[pid] = p.get(f"{prefix}player_name", nm)
    return ids, names

if isinstance(dg_field, list) and len(dg_field) > 0:
    player_ids, player_names = _extract_field(dg_field)
    if not player_ids:
        print(f"⚠ Could not parse field entries. Sample keys: {list(dg_field[0].keys())}")
else:
    recent_cutoff = pd.Timestamp.now() - pd.Timedelta(days=90)
    recent_players = rounds_df[rounds_df["date"] >= recent_cutoff]
    player_ids = recent_players.groupby("player_id")["sg_total"].mean().nlargest(156).index.tolist()
    player_names = {}

# Deduplicate while preserving order
player_ids = list(dict.fromkeys(player_ids))

print(f"Event: {TARGET_EVENT_NAME} (ID={TARGET_EVENT_ID})")
print(f"Field: {len(player_ids)} players")

# Show top 10 DG predictions
# The "win" field contains DECIMAL ODDS (e.g., 8.0 = +700, 10000.0 = longshot).
# Convert to probability with 1/win. Sort ascending: lowest odds = highest prob.
if isinstance(dg_field, list) and len(dg_field) > 0:
    print(f"\nDataGolf top 10 (for reference):")
    for p in sorted(dg_field, key=lambda x: x.get("win") or 99999)[:10]:
        win_odds = p.get("win") or 0
        win_pct = (100.0 / win_odds) if win_odds > 0 else 0
        disp = p.get("player_name") or p.get("team_name") or "?"
        print(f"  {disp:40s}  DG win prob: {win_pct:.2f}%  (odds: {win_odds:.0f})")


In [ ]:
# --- Step 4: Generate Predictions (with H2H pairwise probabilities) ---
_T_SCALE = float(np.sqrt((cfg.OBSERVATION_DF - 2) / cfg.OBSERVATION_DF))
_SG_SUB_COLS = ["sg_ott", "sg_app", "sg_arg", "sg_putt"]
_SG_EWMA_COLS = ["ewma_sg_ott", "ewma_sg_app", "ewma_sg_arg", "ewma_sg_putt"]

feat_index = player_features.set_index("player_id")
pop_mu = float(player_features["ewma_sg_total"].mean())

if "ewma_sg_total_within_var" in player_features.columns:
    pop_within_var = player_features["ewma_sg_total_within_var"].dropna()
    pop_observed_sigma = float(np.sqrt(pop_within_var.median())) if len(pop_within_var) > 0 else 2.75
else:
    pop_observed_sigma = 2.75
pop_sim_sigma = pop_observed_sigma * _T_SCALE

# Recent form (last 8 rounds per player)
recent_form = (
    rounds_df[rounds_df["player_id"].isin(player_ids)]
    .sort_values("date")
    .groupby("player_id")
    .tail(8)
    .groupby("player_id")["sg_total"]
    .agg(["mean", "count"])
)

# Course-specific SG weights
has_sub = all(c in rounds_df.columns for c in _SG_SUB_COLS)
course_weights = None
if has_sub:
    event_hist = rounds_df[rounds_df["event_id"] == TARGET_EVENT_ID]
    sub_data = event_hist[[c for c in _SG_SUB_COLS if c in event_hist.columns]].dropna()
    if len(sub_data) >= 40:
        variances = sub_data.var()
        total = variances.sum()
        if total > 0.01:
            course_weights = variances / total

# Build player abilities
field = []
for pid in player_ids:
    if pid in feat_index.index:
        row = feat_index.loc[pid]
        mu_mean = float(row["ewma_sg_total"])

        # Course-specific SG weighting
        has_ewma_sub = all(c in feat_index.columns for c in _SG_EWMA_COLS)
        if course_weights is not None and has_ewma_sub:
            try:
                sub_skills = np.array([float(row[c]) for c in _SG_EWMA_COLS])
                if not np.any(np.isnan(sub_skills)):
                    weights = np.array([float(course_weights[c]) for c in _SG_SUB_COLS])
                    mu_course = 4.0 * float(np.dot(weights, sub_skills))
                    if np.isfinite(mu_course):
                        mu_mean = 0.5 * mu_mean + 0.5 * mu_course
            except (KeyError, TypeError):
                pass

        # Recent form blend
        if pid in recent_form.index:
            rf = recent_form.loc[pid]
            if int(rf["count"]) >= 6:
                recent_mu = float(rf["mean"])
                if np.isfinite(recent_mu):
                    mu_mean = 0.6 * mu_mean + 0.4 * recent_mu

        within_var = row.get("ewma_sg_total_within_var", np.nan)
        sigma = float(np.sqrt(within_var)) * _T_SCALE if pd.notna(within_var) and within_var > 0.1 else pop_sim_sigma
        sg_var = max(float(row.get("ewma_sg_total_var", 1.0)), 0.25)
        ess = max(float(row.get("effective_sample_size", 5.0)), 1.0)
        mu_std = min(float(np.sqrt(sg_var / ess)), 0.10)

        if not np.isfinite(mu_mean):
            mu_mean = pop_mu
            mu_std = 0.10
    else:
        mu_mean = pop_mu
        mu_std = 0.10
        sigma = pop_sim_sigma

    field.append(PlayerAbility(
        player_id=pid, mu_mean=mu_mean, mu_std=mu_std, sigma=sigma,
    ))

# Simulate with H2H
sim = MonteCarloSimulator(cfg)
tourney = TournamentConfig(
    event_id=TARGET_EVENT_ID,
    event_name=TARGET_EVENT_NAME,
    field_size=len(field),
    cut_rule=cfg.CUT_RULE,
)

result = sim.simulate_tournament(field, tourney, n_simulations=50_000, compute_h2h=True)
results_df = sim.results_to_dataframe(result)

print(f"{'='*60}")
print(f"PREDICTIONS: {TARGET_EVENT_NAME}")
print(f"{'='*60}")
print(f"Simulations: {result.n_simulations:,}")
print(f"H2H pairs computed: {sum(len(v) for v in result.h2h_probs.values()) // 2:,}")
print(f"\nTop 20 (outright win probability):")
print(results_df.head(20).to_string(index=False))

In [ ]:
# --- Step 5: Pull Matchup Odds (All Books) ---
# Fetch odds from every available bookmaker so we can detect edges
# against soft books (DraftKings, FanDuel, etc.), not just Pinnacle.
from data.api_client import DataGolfClient

client = DataGolfClient(cfg, cache_responses=False)
matchup_odds = client.get_live_matchup_odds(tour="pga")

# If live API has no matchups, fall back to historical file (dry-run)
if matchup_odds.empty:
    print("\n⚠ No live matchup odds. Falling back to historical files...")
    for hist_file in ["matchup_odds_2026.csv", "matchup_odds_2025.csv"]:
        hist_path = cfg.DATA_DIR / hist_file
        if not hist_path.exists():
            continue
        hist = pd.read_csv(hist_path, low_memory=False)
        hist_event = hist[hist["event_id"] == TARGET_EVENT_ID].copy()
        if len(hist_event) == 0:
            continue
        print(f"  Found {len(hist_event)} matchup rows in {hist_file}")
        rows = []
        for _, row in hist_event.iterrows():
            rows.append({
                "p1_dg_id": int(row["p1_dg_id"]),
                "p1_player_name": row.get("p1_player_name", ""),
                "p2_dg_id": int(row["p2_dg_id"]),
                "p2_player_name": row.get("p2_player_name", ""),
                "book": row.get("bookmaker", "pinnacle"),
                "p1_odds": float(row.get("p1_close", row.get("p1_open", 1.91))),
                "p2_odds": float(row.get("p2_close", row.get("p2_open", 1.91))),
            })
        matchup_odds = pd.DataFrame(rows)
        break
else:
    print(f"Live API event: {matchup_odds['event_name'].iloc[0]}")

# Bail out cleanly if still no data (e.g. team events, or event not in history)
if matchup_odds.empty or "book" not in matchup_odds.columns:
    print("\n⚠ No matchup odds available for this event (live API empty and historical fallback had no rows).")
    print("  This is expected for team events (e.g. Zurich Classic) or unscheduled weeks.")
    matchup_all = pd.DataFrame(columns=["p1_dg_id","p1_player_name","p2_dg_id","p2_player_name","book","p1_odds","p2_odds"])
else:
    # Drop datagolf benchmark odds — not a real book
    matchup_all = matchup_odds[matchup_odds["book"].str.lower() != "datagolf"].copy()

    # --- Field confirmation filter ---
    # Remove matchups where either player is not in the confirmed tournament field.
    # player_ids comes from Step 3 (DataGolf pre-tournament predictions).
    field_set = set(player_ids)
    before_count = len(matchup_all)
    matchup_all = matchup_all[
        matchup_all["p1_dg_id"].astype(int).isin(field_set) &
        matchup_all["p2_dg_id"].astype(int).isin(field_set)
    ].copy()
    filtered_count = before_count - len(matchup_all)
    if filtered_count > 0:
        print(f"\nField filter: removed {filtered_count} lines with non-confirmed players")

if matchup_all.empty:
    print("\n⚠ No matchup odds available after filtering. Downstream betting steps will be skipped.")
else:
    book_counts = matchup_all.groupby("book").size().sort_values(ascending=False)
    unique_matchups = matchup_all.groupby(["p1_dg_id", "p2_dg_id"]).ngroups
    print(f"\nMatchups: {unique_matchups} unique pairs across {len(book_counts)} books")
    print(f"\nBook breakdown:")
    for book, cnt in book_counts.items():
        print(f"  {book:20s} {cnt:4d} lines")
    print(f"\nSample matchups:")
    sample = matchup_all.drop_duplicates(subset=["p1_dg_id", "p2_dg_id"]).head(5)
    for _, row in sample.iterrows():
        print(f"  {row['p1_player_name']:25s} vs {row['p2_player_name']:25s}")


In [ ]:
# --- Step 6: Find Edges & Size Bets (Multi-Book) ---
# Scan every (matchup, book) pair for edges. When multiple books offer
# an edge on the same side of the same matchup, keep the one with the
# best decimal odds (highest payout).
#
# MY_BOOKS filtering: bets are placed only at books you have accounts with.
# The output also shows the best overall line across ALL books so you can
# see what you're leaving on the table.

h2h = result.h2h_probs
min_edge = cfg.MATCHUP_MIN_EDGE
bankroll = cfg.INITIAL_BANKROLL
max_exposure = cfg.MAX_TOURNAMENT_EXPOSURE_PCT * bankroll  # 30% = $1,500

# --- Phase 1: collect ALL candidate bets across books ---
candidates = []

for _, row in matchup_all.iterrows():
    p1_id = int(row["p1_dg_id"])
    p2_id = int(row["p2_dg_id"])
    book = row["book"]

    # Look up model probability
    model_p1 = None
    if p1_id in h2h and p2_id in h2h[p1_id]:
        model_p1 = h2h[p1_id][p2_id]
    elif p2_id in h2h and p1_id in h2h[p2_id]:
        model_p1 = 1.0 - h2h[p2_id][p1_id]
    if model_p1 is None:
        continue

    model_p2 = 1.0 - model_p1

    # Devig market odds (proportional — same as backtest)
    p1_odds = float(row["p1_odds"])
    p2_odds = float(row["p2_odds"])
    if p1_odds <= 1.0 or p2_odds <= 1.0:
        continue
    raw_p1 = 1.0 / p1_odds
    raw_p2 = 1.0 / p2_odds
    ov = raw_p1 + raw_p2
    market_p1 = raw_p1 / ov
    market_p2 = raw_p2 / ov

    # Check edges on both sides
    edge_p1 = model_p1 - market_p1
    edge_p2 = model_p2 - market_p2

    if edge_p1 > min_edge:
        candidates.append({
            "matchup_key": (min(p1_id, p2_id), max(p1_id, p2_id)),
            "matchup": f"{row['p1_player_name']} vs {row['p2_player_name']}",
            "bet_on": row["p1_player_name"],
            "bet_on_id": p1_id,
            "book": book,
            "model_p": model_p1,
            "market_p": market_p1,
            "edge": edge_p1,
            "odds": p1_odds,
        })
    elif edge_p2 > min_edge:
        candidates.append({
            "matchup_key": (min(p1_id, p2_id), max(p1_id, p2_id)),
            "matchup": f"{row['p1_player_name']} vs {row['p2_player_name']}",
            "bet_on": row["p2_player_name"],
            "bet_on_id": p2_id,
            "book": book,
            "model_p": model_p2,
            "market_p": market_p2,
            "edge": edge_p2,
            "odds": p2_odds,
        })

# --- Phase 2: deduplicate — best odds per (matchup, side) ---
# Track best across ALL books (for comparison) and best from MY_BOOKS (for betting)
best_overall = {}   # best line across every book
best_my_books = {}  # best line from MY_BOOKS only

for c in candidates:
    key = (c["matchup_key"], c["bet_on_id"])

    # Global best (all books)
    if key not in best_overall or c["odds"] > best_overall[key]["odds"]:
        best_overall[key] = c

    # MY_BOOKS best (only books you can actually bet at)
    if MY_BOOKS is None or c["book"].lower() in [b.lower() for b in MY_BOOKS]:
        if key not in best_my_books or c["odds"] > best_my_books[key]["odds"]:
            best_my_books[key] = c

# --- Phase 3: Kelly sizing (using MY_BOOKS odds) ---
bets = []
for key, c in best_my_books.items():
    model_p = c["model_p"]
    dec_odds = c["odds"]
    b = dec_odds - 1.0
    q = 1.0 - model_p
    kf = max((model_p * b - q) / b, 0) * cfg.KELLY_FRACTION
    kf = min(kf, cfg.MATCHUP_MAX_BET_PCT)
    stake = kf * bankroll
    if stake < 1.0:
        continue
    ev = stake * (model_p * (dec_odds - 1) - q)

    # Look up best overall line for comparison
    glob = best_overall.get(key, c)
    same_book = (glob["book"].lower() == c["book"].lower())

    bets.append({
        "matchup": c["matchup"],
        "bet_on": c["bet_on"],
        "bet_on_id": c["bet_on_id"],
        "book": c["book"],
        "model_p": round(model_p, 3),
        "market_p": round(c["market_p"], 3),
        "edge": round(c["edge"], 3),
        "odds": dec_odds,
        "kelly_frac": round(kf, 4),
        "stake": round(stake, 2),
        "ev": round(ev, 2),
        # Comparison columns: best line across ALL books
        "best_book": glob["book"] if not same_book else "",
        "best_odds": glob["odds"] if not same_book else np.nan,
    })

# Sort by EV and enforce tournament exposure cap
all_bets_df = pd.DataFrame(bets).sort_values("ev", ascending=False) if bets else pd.DataFrame()

if len(all_bets_df) > 0:
    cumulative_stake = 0.0
    selected_idx = []
    for idx, row in all_bets_df.iterrows():
        if cumulative_stake + row["stake"] > max_exposure:
            continue
        selected_idx.append(idx)
        cumulative_stake += row["stake"]
    bets_df = all_bets_df.loc[selected_idx].copy()
else:
    bets_df = all_bets_df

# --- Display ---
books_label = ", ".join(MY_BOOKS) if MY_BOOKS else "ALL"
print(f"{'='*80}")
print(f"RECOMMENDED BETS: {TARGET_EVENT_NAME}")
print(f"{'='*80}")
print(f"Your books:         {books_label}")
print(f"Min edge threshold: {min_edge*100:.0f}%")
print(f"Kelly fraction:     {cfg.KELLY_FRACTION}")
print(f"Bankroll:           ${bankroll:,.2f}")
print(f"Max exposure:       ${max_exposure:,.2f} ({cfg.MAX_TOURNAMENT_EXPOSURE_PCT*100:.0f}% of bankroll)")
print(f"Candidates found:   {len(candidates)} across all books")

# Show how many edges exist at your books vs overall
n_overall = len(best_overall)
n_yours = len(best_my_books)
n_missed = n_overall - n_yours
print(f"Edges (all books):  {n_overall} unique (matchup, side)")
print(f"Edges (your books): {n_yours} unique (matchup, side)")
if n_missed > 0 and MY_BOOKS is not None:
    print(f"  ⚠ {n_missed} edges only available at books you don't have")

print(f"After Kelly filter: {len(all_bets_df)}")
print(f"Selected (EV cap):  {len(bets_df)}")

if len(bets_df) > 0:
    print(f"Total stake:        ${bets_df['stake'].sum():,.2f}")
    print(f"Total EV:           ${bets_df['ev'].sum():,.2f}")
    print(f"Avg edge:           {bets_df['edge'].mean()*100:.1f}%")
    print(f"Avg odds:           {bets_df['odds'].mean():.2f}")
    print(f"Stake range:        ${bets_df['stake'].min():,.2f} – ${bets_df['stake'].max():,.2f}")

    # Book breakdown
    print(f"\nBets by book:")
    for book, grp in bets_df.groupby("book"):
        print(f"  {book:20s} {len(grp):3d} bets  ${grp['stake'].sum():7.2f} staked")

    # Show how much EV you're leaving on the table
    has_better = bets_df["best_book"] != ""
    if has_better.any():
        n_better = has_better.sum()
        print(f"\n  ℹ {n_better}/{len(bets_df)} bets have a better line at another book (see best_book / best_odds columns)")

    print(f"\n{'='*80}")
    # Display — show best_book/best_odds only when they differ
    display_cols = ["matchup", "bet_on", "bet_on_id", "book", "model_p", "market_p",
                    "edge", "odds", "kelly_frac", "stake", "ev", "best_book", "best_odds"]
    out = bets_df[display_cols].copy()
    out["best_odds"] = out["best_odds"].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "")
    print(out.to_string(index=False))
else:
    print("\nNo edges found above threshold. Consider lowering MATCHUP_MIN_EDGE.")

### News Research Briefing (Read-Only)
Researches recent news about players in identified matchup bets. This is **purely informational** — it does NOT modify bets, probabilities, or sizing. Review the briefing before placing bets.

Requires `ANTHROPIC_API_KEY` in `.env`. Toggle with `cfg.NEWS_AGENT_ENABLED`.

In [ ]:
# --- Step 6a: News Research Briefing (READ-ONLY) ---
# Researches recent news about players in your recommended bets.
# This NEVER modifies bets or decisions — it just surfaces information.
# Review the briefing, then decide whether to place each bet.

from research.news_agent import NewsResearchAgent

if len(bets_df) > 0 and cfg.NEWS_AGENT_ENABLED:
    news_agent = NewsResearchAgent(settings=cfg)
    research_report = news_agent.research_bets(
        bets_df=bets_df,
        tournament_name=TARGET_EVENT_NAME,
        event_id=TARGET_EVENT_ID,
    )
    news_agent.display_report(research_report)
else:
    if len(bets_df) == 0:
        print("No bets to research.")
    else:
        print("News agent disabled in settings (cfg.NEWS_AGENT_ENABLED = False).")

In [ ]:
# --- Step 6b: Log Bets to Persistent CSV ---
# Appends this week's bets to h2h_bet_log.csv. Run ONCE per event after placing bets.

from pathlib import Path

BET_LOG_PATH = cfg.OUTPUTS_DIR / "h2h_bet_log.csv"
BET_LOG_COLS = [
    "date_placed", "event_id", "event_name", "matchup", "bet_on", "bet_on_id",
    "book", "model_p", "market_p", "edge", "odds", "kelly_frac", "stake", "ev",
    "settled", "result", "pnl", "settled_date",
]

if len(bets_df) == 0:
    print("No bets to log.")
else:
    # Load existing log to check for duplicates
    existing = pd.DataFrame()
    if BET_LOG_PATH.exists():
        existing = pd.read_csv(BET_LOG_PATH)

    # Build new rows
    new_rows = []
    skipped = 0
    for _, bet in bets_df.iterrows():
        # Check duplicate: same event + matchup + bet_on + book
        if len(existing) > 0:
            dup_mask = (
                (existing["event_id"] == TARGET_EVENT_ID) &
                (existing["matchup"] == bet["matchup"]) &
                (existing["bet_on"] == bet["bet_on"])
            )
            if "book" in existing.columns:
                dup_mask &= (existing["book"] == bet["book"])
            if dup_mask.any():
                skipped += 1
                continue

        new_rows.append({
            "date_placed": datetime.now().strftime("%Y-%m-%d"),
            "event_id": TARGET_EVENT_ID,
            "event_name": TARGET_EVENT_NAME,
            "matchup": bet["matchup"],
            "bet_on": bet["bet_on"],
            "bet_on_id": int(bet["bet_on_id"]),
            "book": bet["book"],
            "model_p": bet["model_p"],
            "market_p": bet["market_p"],
            "edge": bet["edge"],
            "odds": bet["odds"],
            "kelly_frac": bet["kelly_frac"],
            "stake": bet["stake"],
            "ev": bet["ev"],
            "settled": False,
            "result": "",
            "pnl": 0.0,
            "settled_date": "",
        })

    if new_rows:
        new_df = pd.DataFrame(new_rows, columns=BET_LOG_COLS)
        if len(existing) > 0:
            combined = pd.concat([existing, new_df], ignore_index=True)
        else:
            combined = new_df
        combined.to_csv(BET_LOG_PATH, index=False)
        print(f"Logged {len(new_rows)} new bets for {TARGET_EVENT_NAME}")
    else:
        print(f"All bets already logged (skipped {skipped} duplicates)")

    # Show totals
    log = pd.read_csv(BET_LOG_PATH)
    n_events = log["event_id"].nunique()
    n_settled = log["settled"].sum()
    n_unsettled = len(log) - n_settled
    print(f"\nBet log: {len(log)} total bets across {n_events} events")
    print(f"  Settled: {int(n_settled)}  |  Unsettled: {int(n_unsettled)}")
    if "book" in log.columns:
        print(f"  Books: {sorted(log['book'].dropna().unique())}")
    print(f"  File: {BET_LOG_PATH}")

In [ ]:
# --- Step 7: Validate Against Actual Outcomes (Dry Run) ---
# Determines H2H outcomes from actual tournament finishing positions.
# Works for ANY matchup pairing — not limited to DataGolf-tracked matchups.
# Lower total score = winner. CUT/WD/DQ loses to anyone who made the cut.
# Both missed cut = compare 2-round totals.

# Load actual round scores for the target event
actuals_rounds = None
for sg_file in ["sg_rounds_pga_2026.csv", "sg_rounds_pga_2025.csv"]:
    sg_path = cfg.DATA_DIR / sg_file
    if sg_path.exists():
        tmp = pd.read_csv(sg_path, low_memory=False)
        tmp_event = tmp[tmp["event_id"] == TARGET_EVENT_ID]
        if len(tmp_event) > 0:
            actuals_rounds = tmp_event.copy()
            print(f"Using results from {sg_file}")
            break

if actuals_rounds is not None and len(bets_df) > 0:
    # Compute total score per player
    player_results = actuals_rounds.groupby(["dg_id", "player_name"]).agg(
        total_score=("round_score", "sum"),
        rounds_played=("round_num", "count"),
        fin_text=("fin_text", "first"),
    ).reset_index()

    def h2h_outcome(p1_row, p2_row):
        """Determine H2H winner from tournament results.
        Returns 'p1', 'p2', or 'tie'."""
        missed = {"CUT", "WD", "DQ", "MDF"}
        p1_miss = p1_row["fin_text"] in missed
        p2_miss = p2_row["fin_text"] in missed
        if p1_miss and not p2_miss:
            return "p2"
        if not p1_miss and p2_miss:
            return "p1"
        # Both made cut or both missed — compare total scores
        if p1_row["total_score"] < p2_row["total_score"]:
            return "p1"
        elif p1_row["total_score"] > p2_row["total_score"]:
            return "p2"
        return "tie"

    # Parse opponent name from matchup string
    def get_opponent_name(matchup_str, bet_on_name):
        parts = matchup_str.split(" vs ")
        if len(parts) != 2:
            return None
        a, b = parts[0].strip(), parts[1].strip()
        return b if a == bet_on_name else a

    wins, losses, ties, no_data = 0, 0, 0, 0
    total_pnl = 0.0
    results = []

    for _, bet in bets_df.iterrows():
        bet_on = bet["bet_on"]
        bet_id = int(bet["bet_on_id"])
        opp_name = get_opponent_name(bet["matchup"], bet_on)
        odds = float(bet["odds"])
        stake = float(bet["stake"])

        # Look up both players in tournament results
        bet_row = player_results[player_results["dg_id"] == bet_id]
        opp_row = player_results[player_results["player_name"] == opp_name] if opp_name else pd.DataFrame()

        if len(bet_row) == 0 or len(opp_row) == 0:
            missing = bet_on if len(bet_row) == 0 else opp_name
            no_data += 1
            results.append({"bet": bet["matchup"], "bet_on": bet_on,
                           "result": "NO DATA", "pnl": 0.0,
                           "detail": f"{missing} not in event"})
            continue

        br, opr = bet_row.iloc[0], opp_row.iloc[0]
        outcome = h2h_outcome(br, opr)

        detail = (f"{bet_on}: {br['fin_text']} ({br['total_score']:.0f}) vs "
                  f"{opp_name}: {opr['fin_text']} ({opr['total_score']:.0f})")

        if outcome == "p1":  # bet_on wins
            profit = stake * (odds - 1)
            total_pnl += profit
            wins += 1
            results.append({"bet": bet["matchup"], "bet_on": bet_on,
                           "result": "WIN", "pnl": round(profit, 2),
                           "detail": detail})
        elif outcome == "p2":  # opponent wins
            total_pnl -= stake
            losses += 1
            results.append({"bet": bet["matchup"], "bet_on": bet_on,
                           "result": "LOSS", "pnl": round(-stake, 2),
                           "detail": detail})
        else:  # tie = void (stake returned)
            ties += 1
            results.append({"bet": bet["matchup"], "bet_on": bet_on,
                           "result": "TIE/VOID", "pnl": 0.0,
                           "detail": detail})

    results_df_val = pd.DataFrame(results)
    total_staked = bets_df["stake"].sum()
    settled = wins + losses

    print(f"\n{'='*100}")
    print(f"DRY RUN RESULTS: {TARGET_EVENT_NAME}")
    print(f"{'='*100}")
    if settled > 0:
        print(f"Wins: {wins}  |  Losses: {losses}  |  Ties: {ties}  |  "
              f"No data: {no_data}  |  Win rate: {wins/settled*100:.1f}%")
    else:
        print("No settled bets")
    print(f"Total staked: ${total_staked:,.2f}")
    if total_staked > 0:
        print(f"P&L: ${total_pnl:,.2f}  |  ROI: {total_pnl/total_staked*100:.1f}%")
    print(f"\n{results_df_val.to_string(index=False)}")
else:
    print("No tournament results found or no bets to validate.")


In [ ]:
# --- Step 7b: Settle Bets in Persistent Log ---
# Run AFTER the tournament finishes. Pulls outcomes and updates the bet log.
# First re-run script 13 to get latest matchup outcomes, then run this cell.

BET_LOG_PATH = cfg.OUTPUTS_DIR / "h2h_bet_log.csv"

if not BET_LOG_PATH.exists():
    print("No bet log found. Run Step 6b first.")
else:
    log = pd.read_csv(BET_LOG_PATH)
    unsettled = log[log["settled"] == False].copy()

    if len(unsettled) == 0:
        print("All bets already settled.")
    else:
        print(f"Unsettled bets: {len(unsettled)} across events: "
              f"{sorted(unsettled['event_id'].unique())}")

        # Load outcome data from historical matchup files
        outcomes_loaded = {}
        for hist_file in ["matchup_odds_2026.csv", "matchup_odds_2025.csv"]:
            hist_path = cfg.DATA_DIR / hist_file
            if hist_path.exists():
                hist = pd.read_csv(hist_path, low_memory=False)
                for eid in unsettled["event_id"].unique():
                    if eid in outcomes_loaded:
                        continue
                    event_data = hist[hist["event_id"] == eid]
                    if len(event_data) > 0:
                        # Check if outcomes are populated
                        has_outcomes = event_data["p1_outcome"].notna().any()
                        if has_outcomes:
                            outcomes_loaded[eid] = event_data
                            print(f"  Loaded {len(event_data)} outcomes for event {eid} from {hist_file}")

        settled_count = 0
        for idx, bet in unsettled.iterrows():
            eid = bet["event_id"]
            if eid not in outcomes_loaded:
                continue

            actuals = outcomes_loaded[eid]
            bet_id = int(bet["bet_on_id"])

            # Find matching matchup
            matched = actuals[
                (actuals["p1_dg_id"] == bet_id) | (actuals["p2_dg_id"] == bet_id)
            ]

            for _, m in matched.iterrows():
                p1, p2 = int(m["p1_dg_id"]), int(m["p2_dg_id"])
                opp_name = m["p2_player_name"] if p1 == bet_id else m["p1_player_name"]

                if opp_name not in bet["matchup"]:
                    continue

                p1_out = m.get("p1_outcome", "")
                winner_id = p1 if p1_out == 1.0 else (p2 if p1_out == 0.0 else None)

                if winner_id is None:
                    log.at[idx, "result"] = "VOID"
                    log.at[idx, "pnl"] = 0.0
                elif winner_id == bet_id:
                    profit = bet["stake"] * (bet["odds"] - 1)
                    log.at[idx, "result"] = "WIN"
                    log.at[idx, "pnl"] = round(profit, 2)
                else:
                    log.at[idx, "result"] = "LOSS"
                    log.at[idx, "pnl"] = round(-bet["stake"], 2)

                log.at[idx, "settled"] = True
                log.at[idx, "settled_date"] = datetime.now().strftime("%Y-%m-%d")
                settled_count += 1
                break

        log.to_csv(BET_LOG_PATH, index=False)

        # Summary
        settled_log = log[log["settled"] == True]
        wins = (settled_log["result"] == "WIN").sum()
        losses = (settled_log["result"] == "LOSS").sum()
        voids = (settled_log["result"] == "VOID").sum()
        total_pnl = settled_log["pnl"].sum()
        total_staked = settled_log["stake"].sum()

        print(f"\nSettled {settled_count} bets this run")
        print(f"\n{'='*50}")
        print(f"LIFETIME RESULTS")
        print(f"{'='*50}")
        print(f"Total bets:    {len(settled_log)} settled + {len(log) - len(settled_log)} pending")
        print(f"Wins:          {wins}")
        print(f"Losses:        {losses}")
        print(f"Voids:         {voids}")
        if wins + losses > 0:
            print(f"Win rate:      {wins/(wins+losses)*100:.1f}%")
        print(f"Total staked:  ${total_staked:,.2f}")
        print(f"Total P&L:     ${total_pnl:,.2f}")
        if total_staked > 0:
            print(f"ROI:           {total_pnl/total_staked*100:.1f}%")
        print(f"Bankroll:      ${cfg.INITIAL_BANKROLL + total_pnl:,.2f}")

In [ ]:
# --- Step 8: Diagnostic Visualizations ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f"H2H Matchup Betting Diagnostics — {TARGET_EVENT_NAME}", fontsize=14, fontweight="bold")

# Need results_df_val from Step 7
settled = results_df_val[results_df_val["result"].isin(["WIN", "LOSS"])].copy()
settled = settled.merge(bets_df[["matchup", "bet_on", "model_p", "market_p", "edge", "odds", "stake"]],
                        left_on=["bet", "bet_on"], right_on=["matchup", "bet_on"], how="left")

# --- 1. Cumulative P&L ---
ax = axes[0, 0]
cum_pnl = results_df_val["pnl"].cumsum()
colors_line = ["green" if v >= 0 else "red" for v in cum_pnl]
ax.plot(range(len(cum_pnl)), cum_pnl, color="steelblue", linewidth=1.5)
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl >= 0, alpha=0.15, color="green")
ax.fill_between(range(len(cum_pnl)), cum_pnl, 0,
                where=cum_pnl < 0, alpha=0.15, color="red")
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.set_xlabel("Bet #")
ax.set_ylabel("Cumulative P&L ($)")
ax.set_title("Cumulative P&L")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# --- 2. Win Rate by Edge Bucket ---
ax = axes[0, 1]
if len(settled) > 0 and "edge" in settled.columns:
    settled["edge_bucket"] = pd.cut(settled["edge"], bins=[0, 0.10, 0.15, 0.20, 0.30, 1.0],
                                     labels=["8-10%", "10-15%", "15-20%", "20-30%", "30%+"])
    bucket_stats = settled.groupby("edge_bucket", observed=True).agg(
        wins=("result", lambda x: (x == "WIN").sum()),
        total=("result", "count")
    )
    bucket_stats["win_rate"] = bucket_stats["wins"] / bucket_stats["total"]
    bars = ax.bar(range(len(bucket_stats)), bucket_stats["win_rate"], color="steelblue", alpha=0.8)
    ax.set_xticks(range(len(bucket_stats)))
    ax.set_xticklabels(bucket_stats.index, rotation=0)
    for i, (_, row) in enumerate(bucket_stats.iterrows()):
        ax.text(i, row["win_rate"] + 0.02, f"{int(row['wins'])}/{int(row['total'])}",
                ha="center", fontsize=9)
    ax.axhline(0.5, color="red", linewidth=1, linestyle="--", label="Breakeven")
    ax.set_ylabel("Win Rate")
    ax.set_xlabel("Model Edge")
    ax.set_title("Win Rate by Edge Size")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

# --- 3. Edge Distribution (all qualifying bets) ---
ax = axes[0, 2]
if len(all_bets_df) > 0:
    ax.hist(all_bets_df["edge"] * 100, bins=20, color="steelblue", alpha=0.7, edgecolor="white")
    if len(bets_df) > 0:
        ax.axvline(bets_df["edge"].min() * 100, color="orange", linestyle="--",
                   label=f"Selected cutoff ({bets_df['edge'].min()*100:.0f}%)")
    ax.axvline(min_edge * 100, color="red", linestyle="--", label=f"Min edge ({min_edge*100:.0f}%)")
    ax.set_xlabel("Edge (%)")
    ax.set_ylabel("Count")
    ax.set_title(f"Edge Distribution ({len(all_bets_df)} qualifying bets)")
    ax.legend(fontsize=8)

# --- 4. Model Prob vs Market Prob (calibration scatter) ---
ax = axes[1, 0]
if len(settled) > 0 and "model_p" in settled.columns and "market_p" in settled.columns:
    wins = settled[settled["result"] == "WIN"]
    losses = settled[settled["result"] == "LOSS"]
    ax.scatter(wins["market_p"], wins["model_p"], c="green", alpha=0.5, s=30, label=f"Win ({len(wins)})")
    ax.scatter(losses["market_p"], losses["model_p"], c="red", alpha=0.5, s=30, label=f"Loss ({len(losses)})")
    ax.plot([0.3, 0.9], [0.3, 0.9], "k--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Market Prob (devigged)")
    ax.set_ylabel("Model Prob")
    ax.set_title("Model vs Market Probability")
    ax.legend(fontsize=8)
    ax.set_xlim(0.3, 0.9)
    ax.set_ylim(0.3, 0.9)

# --- 5. P&L by Player (most bet-on players) ---
ax = axes[1, 1]
if len(results_df_val) > 0:
    player_pnl = results_df_val.groupby("bet_on")["pnl"].sum().sort_values()
    # Show top/bottom 10
    show = pd.concat([player_pnl.head(5), player_pnl.tail(5)])
    colors_bar = ["red" if v < 0 else "green" for v in show.values]
    # Shorten names for display
    short_names = [n.split(",")[0] if "," in n else n for n in show.index]
    ax.barh(range(len(show)), show.values, color=colors_bar, alpha=0.7)
    ax.set_yticks(range(len(show)))
    ax.set_yticklabels(short_names, fontsize=8)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("P&L ($)")
    ax.set_title("P&L by Player (top/bottom 5)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# --- 6. Summary Stats Box ---
ax = axes[1, 2]
ax.axis("off")
total_bets = len(results_df_val)
settled_count = len(settled)
win_count = (settled["result"] == "WIN").sum() if len(settled) > 0 else 0
loss_count = (settled["result"] == "LOSS").sum() if len(settled) > 0 else 0
void_count = (results_df_val["result"] == "TIE/VOID").sum()
total_pnl = results_df_val["pnl"].sum()
total_staked = bets_df["stake"].sum()
roi = (total_pnl / total_staked * 100) if total_staked > 0 else 0
win_rate = (win_count / settled_count * 100) if settled_count > 0 else 0
avg_win = settled[settled["result"] == "WIN"]["pnl"].mean() if win_count > 0 else 0
avg_loss = settled[settled["result"] == "LOSS"]["pnl"].mean() if loss_count > 0 else 0
max_dd = cum_pnl.cummax() - cum_pnl
max_dd_val = max_dd.max()

stats_text = (
    f"SUMMARY\n"
    f"{'─' * 30}\n"
    f"Total bets:      {total_bets}\n"
    f"  Wins:          {win_count}\n"
    f"  Losses:        {loss_count}\n"
    f"  Ties/Void:     {void_count}\n"
    f"{'─' * 30}\n"
    f"Win rate:        {win_rate:.1f}%\n"
    f"ROI:             {roi:.1f}%\n"
    f"{'─' * 30}\n"
    f"Total staked:    ${total_staked:,.2f}\n"
    f"Total P&L:       ${total_pnl:,.2f}\n"
    f"Avg win:         ${avg_win:,.2f}\n"
    f"Avg loss:        ${avg_loss:,.2f}\n"
    f"Max drawdown:    ${max_dd_val:,.2f}\n"
    f"{'─' * 30}\n"
    f"Avg edge:        {bets_df['edge'].mean()*100:.1f}%\n"
    f"Avg odds:        {bets_df['odds'].mean():.2f}\n"
    f"Kelly fraction:  {cfg.KELLY_FRACTION}\n"
    f"Bankroll:        ${bankroll:,.2f}\n"
)
ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3))

plt.tight_layout()
plt.savefig(str(cfg.OUTPUTS_DIR / f"h2h_diagnostics_{TARGET_EVENT_ID}.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {cfg.OUTPUTS_DIR / f'h2h_diagnostics_{TARGET_EVENT_ID}.png'}")

In [ ]:
# --- Step 9: Lifetime Performance Dashboard ---
# Reads the persistent h2h_bet_log.csv and visualizes cross-event performance.
# Run anytime to see cumulative stats across all tracked tournaments.

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

BET_LOG_PATH = cfg.OUTPUTS_DIR / "h2h_bet_log.csv"

if not BET_LOG_PATH.exists():
    print("No bet log found. Run Step 6b first to start logging bets.")
else:
    log = pd.read_csv(BET_LOG_PATH)
    settled = log[log["settled"] == True].copy()

    if len(settled) < 2:
        print(f"Only {len(settled)} settled bets — need at least 2 for dashboard.")
        print(f"Total bets in log: {len(log)} ({len(log) - len(settled)} unsettled)")
    else:
        # Parse dates
        settled["date_placed"] = pd.to_datetime(settled["date_placed"])
        settled["settled_date"] = pd.to_datetime(settled["settled_date"])
        settled = settled.sort_values("settled_date").reset_index(drop=True)

        # Cumulative P&L
        settled["cum_pnl"] = settled["pnl"].cumsum()
        settled["cum_stake"] = settled["stake"].cumsum()
        settled["bet_num"] = range(1, len(settled) + 1)

        # Rolling win rate (50-bet window, or all if < 50)
        window = min(50, len(settled))
        settled["is_win"] = (settled["result"] == "WIN").astype(int)
        settled["rolling_wr"] = settled["is_win"].rolling(window, min_periods=5).mean() * 100

        # Bankroll curve
        settled["bankroll"] = cfg.INITIAL_BANKROLL + settled["cum_pnl"]

        # Monthly P&L
        settled["month"] = settled["settled_date"].dt.to_period("M")
        monthly = settled.groupby("month").agg(
            pnl=("pnl", "sum"),
            bets=("pnl", "count"),
            staked=("stake", "sum"),
        ).reset_index()
        monthly["month_str"] = monthly["month"].astype(str)
        monthly["roi"] = (monthly["pnl"] / monthly["staked"] * 100).round(1)

        # ROI by event
        by_event = settled.groupby("event_name").agg(
            pnl=("pnl", "sum"),
            bets=("pnl", "count"),
            staked=("stake", "sum"),
            wins=("is_win", "sum"),
        ).reset_index()
        by_event["roi"] = (by_event["pnl"] / by_event["staked"] * 100).round(1)
        by_event["wr"] = (by_event["wins"] / by_event["bets"] * 100).round(1)

        # --- Summary stats ---
        total_bets = len(settled)
        wins = settled["is_win"].sum()
        losses = (settled["result"] == "LOSS").sum()
        voids = (settled["result"] == "VOID").sum()
        win_rate = wins / (wins + losses) * 100 if (wins + losses) > 0 else 0
        total_staked = settled["stake"].sum()
        total_pnl = settled["pnl"].sum()
        roi = total_pnl / total_staked * 100 if total_staked > 0 else 0
        current_bankroll = cfg.INITIAL_BANKROLL + total_pnl

        # Max drawdown
        cum_max = settled["cum_pnl"].cummax()
        drawdown = settled["cum_pnl"] - cum_max
        max_dd = drawdown.min()

        # Sharpe-like ratio (daily P&L std)
        if len(settled) > 1:
            daily_pnl = settled.groupby("settled_date")["pnl"].sum()
            sharpe = (daily_pnl.mean() / daily_pnl.std() * np.sqrt(52)) if daily_pnl.std() > 0 else 0
        else:
            sharpe = 0

        # === PLOT ===
        fig = plt.figure(figsize=(18, 14))
        gs = GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)
        fig.suptitle("Lifetime H2H Betting Performance", fontsize=16, fontweight="bold", y=0.98)

        # 1. Cumulative P&L over time
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.plot(settled["bet_num"], settled["cum_pnl"], color="#2196F3", linewidth=2)
        ax1.fill_between(settled["bet_num"], 0, settled["cum_pnl"],
                         where=settled["cum_pnl"] >= 0, alpha=0.15, color="#4CAF50")
        ax1.fill_between(settled["bet_num"], 0, settled["cum_pnl"],
                         where=settled["cum_pnl"] < 0, alpha=0.15, color="#f44336")
        ax1.axhline(0, color="gray", linewidth=0.8, linestyle="--")
        ax1.set_xlabel("Bet #")
        ax1.set_ylabel("Cumulative P&L ($)")
        ax1.set_title("Cumulative P&L")
        ax1.grid(True, alpha=0.3)

        # 2. Monthly P&L bars
        ax2 = fig.add_subplot(gs[0, 1])
        colors_monthly = ["#4CAF50" if x >= 0 else "#f44336" for x in monthly["pnl"]]
        bars = ax2.bar(monthly["month_str"], monthly["pnl"], color=colors_monthly, edgecolor="white")
        for bar, roi_val in zip(bars, monthly["roi"]):
            y = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2, y + (5 if y >= 0 else -15),
                     f"{roi_val:+.0f}%", ha="center", va="bottom" if y >= 0 else "top",
                     fontsize=8, fontweight="bold")
        ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
        ax2.set_ylabel("P&L ($)")
        ax2.set_title("Monthly P&L")
        ax2.tick_params(axis="x", rotation=45)
        ax2.grid(True, alpha=0.3, axis="y")

        # 3. Rolling win rate
        ax3 = fig.add_subplot(gs[1, 0])
        ax3.plot(settled["bet_num"], settled["rolling_wr"], color="#FF9800", linewidth=2)
        ax3.axhline(50, color="gray", linewidth=0.8, linestyle="--", label="Break-even")
        ax3.set_xlabel("Bet #")
        ax3.set_ylabel("Win Rate (%)")
        ax3.set_title(f"Rolling Win Rate ({window}-bet window)")
        ax3.set_ylim(30, 80)
        ax3.legend(fontsize=8)
        ax3.grid(True, alpha=0.3)

        # 4. ROI by event
        ax4 = fig.add_subplot(gs[1, 1])
        evt_colors = ["#4CAF50" if x >= 0 else "#f44336" for x in by_event["roi"]]
        evt_labels = [n[:20] for n in by_event["event_name"]]
        bars4 = ax4.barh(evt_labels, by_event["roi"], color=evt_colors, edgecolor="white")
        for bar, wr, n in zip(bars4, by_event["wr"], by_event["bets"]):
            w = bar.get_width()
            ax4.text(w + 1, bar.get_y() + bar.get_height()/2,
                     f"{wr:.0f}% WR ({n})", va="center", fontsize=8)
        ax4.axvline(0, color="gray", linewidth=0.8, linestyle="--")
        ax4.set_xlabel("ROI (%)")
        ax4.set_title("ROI by Event")
        ax4.grid(True, alpha=0.3, axis="x")

        # 5. Bankroll growth curve
        ax5 = fig.add_subplot(gs[2, 0])
        ax5.plot(settled["bet_num"], settled["bankroll"], color="#9C27B0", linewidth=2)
        ax5.axhline(cfg.INITIAL_BANKROLL, color="gray", linewidth=0.8, linestyle="--",
                     label=f"Starting (${cfg.INITIAL_BANKROLL:,.0f})")
        ax5.fill_between(settled["bet_num"], cfg.INITIAL_BANKROLL, settled["bankroll"],
                         where=settled["bankroll"] >= cfg.INITIAL_BANKROLL, alpha=0.1, color="#4CAF50")
        ax5.fill_between(settled["bet_num"], cfg.INITIAL_BANKROLL, settled["bankroll"],
                         where=settled["bankroll"] < cfg.INITIAL_BANKROLL, alpha=0.1, color="#f44336")
        ax5.set_xlabel("Bet #")
        ax5.set_ylabel("Bankroll ($)")
        ax5.set_title("Bankroll Growth")
        ax5.legend(fontsize=8)
        ax5.grid(True, alpha=0.3)

        # 6. Summary stats table
        ax6 = fig.add_subplot(gs[2, 1])
        ax6.axis("off")
        stats_data = [
            ["Total Bets (settled)", f"{total_bets}"],
            ["Record", f"{wins}W - {losses}L - {voids}V"],
            ["Win Rate", f"{win_rate:.1f}%"],
            ["Total Staked", f"${total_staked:,.2f}"],
            ["Total P&L", f"${total_pnl:,.2f}"],
            ["ROI", f"{roi:+.1f}%"],
            ["Current Bankroll", f"${current_bankroll:,.2f}"],
            ["Max Drawdown", f"${max_dd:,.2f}"],
            ["Sharpe (annualized)", f"{sharpe:.2f}"],
            ["Events Tracked", f"{by_event.shape[0]}"],
            ["Unsettled Bets", f"{len(log) - len(settled)}"],
        ]
        table = ax6.table(cellText=stats_data, colLabels=["Metric", "Value"],
                          loc="center", cellLoc="left", colWidths=[0.5, 0.4])
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1.0, 1.6)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_facecolor("#37474F")
                cell.set_text_props(color="white", fontweight="bold")
            elif row % 2 == 0:
                cell.set_facecolor("#f5f5f5")
            cell.set_edgecolor("#e0e0e0")
        ax6.set_title("Lifetime Summary", fontsize=12, fontweight="bold", pad=20)

        # Save
        out_path = cfg.OUTPUTS_DIR / "h2h_lifetime_dashboard.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
        plt.show()
        print(f"\nDashboard saved to {out_path}")


---
## Live Deployment Workflow

**Weekly routine:**
1. Update data (pull latest rounds from DataGolf or CSVs)
2. Set `TARGET_EVENT_ID` and `TARGET_EVENT_NAME` for this week's tournament
3. Run cells 1-4 to generate H2H predictions (~2-3 min for 50K sims)
4. Load matchup odds from sportsbook → uncomment cell 5
5. Run cells 5-6 to find edges and size bets
6. Place bets at sportsbook
7. After tournament: settle bets, track cumulative P&L

**Risk management:**
- Max 2% of bankroll per individual matchup bet
- If 3 consecutive losing months → re-run backtest validation (notebook 08)
- If matchup gates fail on rolling validation → stop betting, investigate
- Monitor win rate — should converge to >52% over 200+ bets